# Store Sales Forecasting - End-to-End Pipeline

This notebook executes data cleaning, feature engineering, saves processed datasets to `data/processed/`, and displays all 9 exploratory graphs from `main.ipynb`.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
sys.path.append("..")

from src.preprocessing.load_data import load_raw_data
from src.preprocessing.holidays import process_holidays
from src.preprocessing.oil import process_oil
from src.preprocessing.transactions import process_transactions
from src.preprocessing.clean_data import filter_pre_opening_days
from src.features.build_features import build_all_features
from src.visualization.plots import plot_all_visualizations
from src.models.train import train_lgbm, train_xgb
from src.models.predict import dirrec_predict
from src.utils.submission import generate_submission

## 1. Data Cleaning & Preprocessing

In [2]:
# ## 1. Data Cleaning & Preprocessing
df, stores_df, transactions_df, oil_df, holidays_df, test_df = load_raw_data("../data/raw")
print("Raw merged df shape:", df.shape)

# 1. Process Holidays
df = process_holidays(df, holidays_df)
test_df = process_holidays(test_df, holidays_df)

# 2. Process Oil Prices
df, full_oil_df = process_oil(df, oil_df)
test_df, _ = process_oil(test_df, oil_df)

# 3. Process Transactions (Linear regression imputation & closed days)
df, daily_sales = process_transactions(df, transactions_df)

# 4. Filter Pre-Opening Zero-Sales Days
df = filter_pre_opening_days(df)
print("Cleaned df shape:", df.shape)

Raw merged df shape: (3000888, 10)
Cleaned df shape: (2778831, 16)


## 2. Feature Engineering & Saving Processed Output

In [3]:
# ## 2. Feature Engineering & Saving Processed Output
df, test_df = build_all_features(df, test_df)

print("Clean Training df Shape:", df.shape)
print("Aligned Test test_df Shape:", test_df.shape)
print("Do test_df and df columns match 100%?", list(test_df.columns) == list(df.columns))

# Save processed dataset
os.makedirs("../data/processed", exist_ok=True)
processed_path = "../data/processed/train_processed.parquet"
df.to_parquet(processed_path, index=False)
print(f"Processed training dataset saved to: {processed_path}")

Clean Training df Shape: (2728935, 42)
Aligned Test test_df Shape: (28512, 42)
Do test_df and df columns match 100%? True
Processed training dataset saved to: ../data/processed/train_processed.parquet


In [4]:
df.isnull().sum()

id                       0
date                     0
store_nbr                0
family                   0
sales                    0
onpromotion              0
city                     0
state                    0
type                     0
cluster                  0
is_Holiday               0
is_Event                 0
oil_price                0
oil_roll_mean_30         0
oil_diff_7               0
transactions             0
day_Monday               0
day_Tuesday              0
day_Wednesday            0
day_Thursday             0
day_Friday               0
day_Saturday             0
day_Sunday               0
is_payday_window         0
mean_sales_by_family     0
mean_sales_by_store      0
mean_sales_by_cluster    0
mean_sales_by_type       0
onpromotion_lead_1       0
family_dollar_diff       0
expected_promo_boost     0
sales_lag_1              0
sales_lag_7              0
sales_lag_14             0
sales_lag_21             0
sales_lag_28             0
sales_roll_mean_30       0
s

In [5]:
test_df.isnull().sum()

id                           0
date                         0
store_nbr                    0
family                       0
sales                    28512
onpromotion                  0
city                         0
state                        0
type                         0
cluster                      0
is_Holiday                   0
is_Event                     0
oil_price                    0
oil_roll_mean_30             0
oil_diff_7                   0
transactions             28512
day_Monday                   0
day_Tuesday                  0
day_Wednesday                0
day_Thursday                 0
day_Friday                   0
day_Saturday                 0
day_Sunday                   0
is_payday_window             0
mean_sales_by_family         0
mean_sales_by_store          0
mean_sales_by_cluster        0
mean_sales_by_type           0
onpromotion_lead_1           0
family_dollar_diff           0
expected_promo_boost         0
sales_lag_1              26730
sales_la

## 3. Exploratory Data Visualizations

In [6]:
# ## 3. Exploratory Data Visualizations (Saved to plots/)
plot_all_visualizations(df, full_oil_df, output_dir="../plots")

Generating and saving all 9 plots to '../plots/'...
All 9 plots saved successfully to '../plots/'!


## 4. Fit 54 Store Linear Models

In [7]:
# ## 4. Fit 54 Store Linear Regression Models for Transaction Estimation
daily_sales_df = df.groupby(["date", "store_nbr"]).agg(
    total_sales=("sales", "sum"),
    transactions=("transactions", "first")
).reset_index()

store_models = {}
for store_nbr in range(1, 55):
    store_data = daily_sales_df[
        (daily_sales_df["store_nbr"] == store_nbr) & 
        (daily_sales_df["transactions"].notnull()) & 
        (daily_sales_df["total_sales"] > 0)
    ]
    x = store_data["total_sales"].values
    y = store_data["transactions"].values
    m, c = np.polyfit(x, y, 1)
    store_models[store_nbr] = (m, c)

print(f"54 Store Linear Regression Models fitted successfully! Total store models: {len(store_models)}")

54 Store Linear Regression Models fitted successfully! Total store models: 54


## 5. Baseline Model Trsining and Evaluation

In [8]:
# ## 5. Baseline Model Training & 16-Day Holdout Validation Split
lgb_model, lgb_score = train_lgbm(df)
xgb_model, xgb_score = train_xgb(df)

print("\n" + "="*50)
print(f"LightGBM Baseline Validation RMSLE Score: {lgb_score:.5f}")
print(f"XGBoost  Baseline Validation RMSLE Score: {xgb_score:.5f}")
print("="*50)

X_train shape: 2,702,205 rows × 39 features
X_val shape:   26,730 rows × 39 features


c:\Users\rohet\OneDrive\Documents\CS_WORK\Data_Science\Store_Sales_Forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.235203 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4814
[LightGBM] [Info] Number of data points in the train set: 2702205, number of used features: 39
[LightGBM] [Info] Start training from score 3.166761
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's l2: 0.132033

Baseline LightGBM Validation RMSLE Score: 0.36335
X_train shape: 2,702,205 rows × 39 features
X_val shape:   26,730 rows × 39 features
[0]	validation_0-rmse:2.48834
[1]	validation_0-rmse:2.41540
[2]	validation_0-rmse:2.34512
[3]	validation_0-rmse:2.27662
[4]	validation_0-rmse:2.21048
[5]	validation_0-rmse:2.14611
[6]	validation_0-rmse:2.08398
[7]	validation_0-rmse:2.02371
[8]	validation_0-rmse:1.96554
[9]	validation_0-rmse:1.90884
[10]	validation_0-

## 6. 16-Day DirRec Forecasting

In [ ]:
# ## 6. 16-Day Direct-Recursive (DirRec) Test Set Forecasting
test_df = dirrec_predict(df, test_df, store_models)

## 7. Generate Kaggle Submission

In [ ]:
# ## 7. Generate Kaggle Submission File
submission = generate_submission(test_df, output_path="../data/processed/submission.csv")